# Advanced Document Intelligence & RAG System
This notebook implements an enterprise-grade Retrieval-Augmented Generation (RAG) system completely from scratch.
By utilizing high-dimensional text sentence mapping and local index structures, this design processes user queries, retrieves mathematically relevant context coordinates, and passes a grounded prompt execution envelope to Google Gemini to completely eliminate hallucinated model data.

### Objective
Develop a simple Retrieval-Augmented Generation (RAG) system to answer questions from custom documents. Build a pipeline that retrieves relevant information from a document and uses a language model to generate answers.

### Dataset
 Use own PDFs:
- Notes
- Resume
- Research papers
- Books
- RAG is meant for custom/private data


### System Pipeline Highlights
1. **Source Content Ingestion**: Multi-tier extraction utility utilizing raw byte page tracking buffers.
2. **Text Clean Filters**: Custom regular expression sanitization to collapse formatting artifacts and structural compression noise.
3. **Semantic Chunk Processing**: Segmenting string frames using explicit multi-boundary character windows while locking permanent page coordinates.
4. **FAISS Vector Matrix Seeding**: Transforming text inputs into dense mathematical vectors and indexing them in an absolute Normalized Flat Inner Product Lookup Database.
5. **Contextual Grounding Pass**: Consolidating matching documentation records into a rigid instruction layout targeting the modern `google-genai` client core.

### Architecture

```text
+---------------------------------------------------------------------------------+
|                         PHASE 1: INDEX COMPILATION                              |
+---------------------------------------------------------------------------------+
|                                                                                 |
|   [ Raw PDF Document ]                                                          |
|            │                                                                    |
|            ▼ (Stage 1: Ingestion & Extraction)                                  |
|   [ Raw Extracted Strings ]                                                     |
|            │                                                                    |
|            ▼ (Stage 2: Regex White-space Cleanup)                               |
|   [ Cleaned Data Stream Pages ]                                                 |
|            │                                                                    |
|            ▼ (Stage 3: Recursive Splitter & Overlap Window)                     |
|   [ Overlapping Text Fragments (Page References Locked) ]                       |
|            │                                                                    |
|            ▼ (Stage 4: Tokenization via Sentence Transformers)                  |
|   [ 384-Dimensional Dense Geometric Tensors ]                                   |
|            │                                                                    |
|            ▼ (Stage 5: L2 Unit Length Normalization)                            |
|   [ Local Flat FAISS Index Storage ] <─── (Persisted Matrix Array)              |
|                                                                                 |
+---------------------------------------------------------------------------------+

+---------------------------------------------------------------------------------+
|                         PHASE 2: INFERENCE EXECUTION                            |
+=================================================================================+
|                                                                                 |
|   [ Incoming User Query ]                                                       |
|            │                                                                    |
|            ▼ (Vectorization via all-MiniLM-L6-v2)                               |
|   [ Query Vector Profile (L2 Normalized) ]                                      |
|            │                                                                    |
|            ├───► [ Flat Cosine-Similarity Search ] ───┐                         |
|            │                      │                   │                         |
|            ▼                      ▼                   ▼                         |
|     (Match Node 1)          (Match Node 2)      (Match Node 3)                  |
|            │                      │                   │                         |
|            └──────────────────────┴───────────────────┴───► [ Evidence Pool ]   |
|                                                                     │           |
|                                                                     ▼           |
|   [ User Query ] ───► [ Strict Grounding Prompt Template Synthesis ] ◄──────┘    |
|                                     │                                           |
|                                     ▼                                           |
|               [ Injected Cloud Payload API Transaction ]                        |
|                                     │                                           |
|                                     ▼                                           |
|               [ Google Gemini 2.5 Flash Generative Pass ]                       |
|                                     │                                           |
|                                     ▼                                           |
|               [ Hallucination-Free Fact-Grounded Response ]                     |
|                                                                                 |
+---------------------------------------------------------------------------------+

###Step 1: INSTALL CORE LIBRARY DEPENDENCIES

In [ ]:
# ============================================================
# STEP 1: INSTALL CORE LIBRARY DEPENDENCIES
# ============================================================
!pip install -q pypdf sentence-transformers faiss-cpu google-genai langchain-text-splitters

###Step 2: Import Libraries (CORE SYSTEM INITIALIZATION)


In [ ]:
# ============================================================
# STEP 2: CORE SYSTEM INITIALIZATION
# ============================================================
import os
import re
import getpass
import numpy as np
import pypdf
import faiss
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google import genai
from google.genai import types

print("[SYSTEM REGISTRATION]: All background modular assets verified and operational.")

### Architectural Component Decoupling

#### 1. The Ingestion Engine (ETL Tier)
* **Extraction**: Intercepts local physical file assets byte-by-byte using `pypdf`.
* **Sanitization**: Implements custom regular expression sweeping array boundaries (`re.sub(r'[ \t]+', ' ', text)`) to prevent visual layouts from adding noise into semantic data strings.

#### 2. Vector Array Indexing Tier
* **Segmentation**: Drops character arrays down using mathematical windows to fit safely within language processing thresholds while ensuring a permanent trace back to raw coordinate tracking frames.
* **Seeding**: Transforms character chains into dense matrices via `SentenceTransformer('all-MiniLM-L6-v2')`. Coordinates undergo immediate $L_2$ unit normalization scaling lengths to 1 to match exact **Cosine Similarity** equations during searching runs.

#### 3. Grounded Cloud Inference Layer
* **Retrieval**: Executes vector matrix operations against `faiss.IndexFlatIP` instances to cache structural evidence data.
* **Grounding Envelope**: Packs retrieved source text structures alongside custom structural validation guardrails to prevent creative model liberties.
* **Inference Pipeline**: Sends the assembled prompt to the production **Gemini 2.5 Flash** model with a hardcoded zero temperature configuration (`temperature=0.0`) to guarantee deterministic factual compliance.

### Step 3: Configure Google GenAI Client
We initialize the new, modern Google GenAI Client wrapper. To prevent accidental security leaks, we check for a local environment variable before falling back to an obfuscated manual password prompt.

In [ ]:
# ============================================================
# STEP 3: SECURE GENAI SDK CREDENTIAL COMPILATION (CORRECTED)
# ============================================================
def establish_ai_session() -> genai.Client:
    """
    Spawns an authorized instance of the official Google GenAI runtime ecosystem.
    """
    runtime_token = os.environ.get("GEMINI_API_KEY")

    if not runtime_token:
        print("[CREDENTIAL WARNING]: 'GEMINI_API_KEY' absent from local environment properties.")
        runtime_token = getpass.getpass("Provide your secret Google Gemini API Gateway Token: ")

    if not runtime_token or not runtime_token.strip():
        # Fixed: Replaced AuthError with ValueError to prevent NameError
        raise ValueError("Pipeline Execution Blocked: Gateway token field value cannot be whitespace.")

    session_client = genai.Client(api_key=runtime_token)
    print("[AUTHENTICATION SUCCESS]: AI API environment framework mapping complete.")
    return session_client

try:
    gemini_client = establish_ai_session()
except Exception as auth_exception:
    print(f"[GATEWAY ERROR] Core registration handler aborted task: {auth_exception}")

### Step 4 & 5: Raw Document Data Stream Reading
We write automated extraction loops to iterate across local PDF structures. The code safely checks for string existence to flag unparseable visual-only formats early.

STEP 4 & 5 : LOAD PDF & EXTRACT RAW MAPPINGS


In [ ]:
# ============================================================
# STEP 4 & 5: LOCAL TARGET DOCUMENT EXTRACTION Routines
# ============================================================
def extract_pdf_page_content(target_file: str) -> list[str]:
    """
    Extracts raw text frames across a physical PDF structure sequence.
    """
    if not os.path.exists(target_file):
        raise FileNotFoundError(f"Operational halt: File item missing at: {target_file}")

    document_reader = pypdf.PdfReader(target_file)
    parsed_page_arrays = []

    for dynamic_page in document_reader.pages:
        extracted_content = dynamic_page.extract_text()
        parsed_page_arrays.append(extracted_content if extracted_content else "")

    if not parsed_page_arrays or all(len(item.strip()) == 0 for item in parsed_page_arrays):
        raise ValueError("Data Stream Fault: Document source frame yields zero parseable data.")

    print(f"[DATA PACKET LOADED]: Found {len(parsed_page_arrays)} valid raw pages.")
    return parsed_page_arrays



###Step 6: ADVANCED TEXT CLEANING NORMALIZATION
Extracted string chunks undergo regex cleaning to collapse tab gaps, align spacing breaks, and suppress vertical line compressions down to predictable paragraphs.

In [ ]:
# ============================================================
# STEP 6: STREAM RE-FORMATTING & WHITESPACE SCRUBBING
# ============================================================
def scrub_text_formatting(raw_data_array: list[str]) -> list[str]:
    """
    Applies custom regular expression patterns to clean layout distortion.
    """
    sanitized_dataset = []
    for item_text in raw_data_array:
        # Collapse excessive spacing or inline tabs into individual space frames
        space_adjusted = re.sub(r'[ \t]+', ' ', item_text)
        # Limit vertical text compression noise while maintaining paragraph blocks
        structure_adjusted = re.sub(r'\n{3,}', '\n\n', space_adjusted)
        sanitized_dataset.append(structure_adjusted.strip())
    return sanitized_dataset

### Step 7: Text Chunking and Metadata Mapping
To protect against the **Context Truncation Bug** and ensure strict document trace capability, texts are chunked via sentence boundaries while locking their respective 1-indexed document source pages.

In [ ]:
# ============================================================
# STEP 7: METADATA-COORDINATED RECURSIVE TEXT BREAKS
# ============================================================
def fragment_text_data(cleaned_source: list[str], slice_size: int = 500, overlap_size: int = 50) -> list[dict]:
    """
    Splits character bodies down into semantically cohesive overlapping segments.
    """
    slicing_orchestrator = RecursiveCharacterTextSplitter(
        chunk_size=slice_size,
        chunk_overlap=overlap_size,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    final_segmented_pool = []
    for sheet_idx, sheet_content in enumerate(cleaned_source):
        if not sheet_content.strip():
            continue

        page_coordinate = sheet_idx + 1 # Align to regular 1-indexed document charts
        text_slices = slicing_orchestrator.split_text(sheet_content)

        for segment_slice in text_slices:
            final_segmented_pool.append({
                "body_content": segment_slice,
                "source_page": page_coordinate
            })

    print(f"[TEXT TRANSFORMATION COMPLETED]: Produced {len(final_segmented_pool)} standalone context tracking units.")
    return final_segmented_pool

### Step 8: Compute Semantic Encodings via Sentence Transformers
We call a local execution instance of the `all-MiniLM-L6-v2` tokenizer architecture to process standard strings into dense 384-dimensional mathematical arrays.

In [ ]:
# ============================================================
# STEP 8: DENSE MATRIX SEMANTIC TRANSFORMATION
# ============================================================
def compile_vector_representations(segmented_dataset: list[dict], transformer_id: str = "all-MiniLM-L6-v2"):
    """
    Transforms string context pools into high-dimensional geometric vector shapes.
    """
    print(f"[TRANSFORMER ACTIVATION]: Querying local text tokenizer model: '{transformer_id}'...")
    local_encoder = SentenceTransformer(transformer_id)

    extracted_strings = [node["body_content"] for node in segmented_dataset]
    vectorized_embeddings = local_encoder.encode(extracted_strings, show_progress_bar=True, convert_to_numpy=True)

    return vectorized_embeddings, local_encoder



### Step 9: Initialize Localized Flat Inner Product Matrix Space
To run exact **Cosine Similarity** equations without performance dropoffs, embedding layers undergo standard $L_2$ unit normalization lengths before populating a Flat Inner Product (`faiss.IndexFlatIP`) lookup database.

In [ ]:
# ============================================================
# STEP 9: INSTANTIATE DENSE HIGH-DIMENSIONAL SEARCH LAYER
# ============================================================
def establish_faiss_vector_index(raw_embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """
    Initializes a database engine optimized for mathematical dot-product correlation evaluations.
    """
    optimized_matrix = np.array(raw_embeddings, dtype=np.float32)

    # Scale lines to absolute unit values to ensure true Cosine Similarity results
    faiss.normalize_L2(optimized_matrix)

    geometric_dimensions = optimized_matrix.shape[1]
    flat_ip_database = faiss.IndexFlatIP(geometric_dimensions)
    flat_ip_database.add(optimized_matrix)

    print(f"[DATABASE ENGINE ONLINE]: Indexed matrix registry holding {flat_ip_database.ntotal} item spaces.")
    return flat_ip_database

### Step 10 & 11: Document Query Retrieval & Grounding Synthesis
User requests undergo immediate mathematical normalization to look up relevant context structures. This returned context is combined with explicit prompt directives to block model hallucinations.

In [ ]:
# ============================================================
# STEP 10 & 11: GEOMETRIC COSINE SIMILARITY RETRIEVAL
# ============================================================
def query_context_store(search_string: str, database: faiss.IndexFlatIP, resource_pool: list[dict], embedding_engine: SentenceTransformer, search_limit: int = 3) -> list[dict]:
    """
    Extracts relevant context pieces based on mathematical vector alignment scores.
    """
    if not search_string.strip():
        raise ValueError("Search Request Aborted: Input string sequence is blank.")

    # Convert query into vector arrays
    query_shape = embedding_engine.encode([search_string], convert_to_numpy=True)
    query_shape_f32 = np.array(query_shape, dtype=np.float32)
    faiss.normalize_L2(query_shape_f32)

    # Perform geometric distance calculation
    similarity_metrics, absolute_positions = database.search(query_shape_f32, search_limit)

    extracted_evidence_nodes = []
    for structural_score, storage_id in zip(similarity_metrics[0], absolute_positions[0]):
        if storage_id < 0 or storage_id >= len(resource_pool):
            continue
        extracted_evidence_nodes.append({
            "text": resource_pool[storage_id]["body_content"],
            "page": resource_pool[storage_id]["source_page"],
            "correlation": float(structural_score)
        })
    return extracted_evidence_nodes



### Step 12 & 13: Prompt Engineering Template Synthesis
Extracted contextual proof records are cleanly formatted alongside custom validation rules. This ensures that the downstream language model outputs answers strictly grounded in your provided text blocks.

In [ ]:
# ============================================================
# STEP 12 & 13: GROUNDED INFERENCE TEMPLATE SYNTHESIS
# ============================================================
def synthesis_grounded_instruction_prompt(user_query: str, factual_contexts: list[dict]) -> str:
    """
    Merges custom constraints and source text structures into an instructional model prompt.
    """
    formatted_data_blocks = []
    for offset_idx, node_item in enumerate(factual_contexts):
        formatted_data_blocks.append(
            f"=== Verified Document Context Reference #{offset_idx + 1} (Location: Page {node_item['page']} | Confidence Score: {node_item['correlation']:.4f}) ===\n"
            f"{node_item['text']}"
        )
    compiled_evidence_stream = "\n\n".join(formatted_data_blocks)

    rigid_prompt_envelope = (
        f"You are an elite enterprise knowledge assistant answering queries strictly from documented proof records.\n"
        f"Operational Directives:\n"
        f"1. Rely EXCLUSIVELY on the verified document context elements provided below.\n"
        f"2. If the context values are missing, insufficient, or unrelated to the prompt, output exactly: "
        f"'Answer not found in the document.'\n"
        f"3. Do not formulate external theories or draw from training assumptions. Stay strictly literal.\n\n"
        f"Verified Document Context:\n"
        f"{compiled_evidence_stream}\n\n"
        f"User Query: {user_query}\n\n"
        f"Answer:"
    )
    return rigid_prompt_envelope

### Step 14: Google Gemini Generation Layer
Using our verified client system instance, the consolidated prompt is dispatched to the high-performance **`gemini-2.5-flash`** engine.

In [ ]:
# ============================================================
# STEP 14: TRANSFORMERS INFERENCE INVOCATION (GEMINI 2.5 FLASH)
# ============================================================
def dispatch_grounded_payload(api_bridge: genai.Client, system_prompt: str) -> str:
    """
    Dispatches constructed grounding blocks to the production Google Gemini model framework.
    """
    try:
        model_transaction = api_bridge.models.generate_content(
            model='gemini-2.5-flash',
            contents=system_prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,  # Suppress creative logic variance to enforce literal facts
                max_output_tokens=450
            )
        )
        return model_transaction.text
    except Exception as inference_fault:
        raise RuntimeError(f"Generative Cloud Engine execution layer rejected task payload: {inference_fault}")

## Phase 1: Preprocessing & Index Assembly (Run Once)
We ask for the PDF location path dynamically, load the target dataset contents, split the string chunks, map vectors, and save the resulting index architecture.

In [ ]:
# ============================================================
# PHASE 1 : DATA PIPELINE EXTRACTION
# ============================================================
user_configured_pdf_path = input("Enter PDF asset path file location : ").strip()

# Local memory space registers
runtime_document_pool = None
runtime_encoder_instance = None
runtime_faiss_matrix = None

if not user_configured_pdf_path:
    print("[PIPELINE CRITICAL EROR]: Target path configuration cannot be whitespace.")
elif not os.path.exists(user_configured_pdf_path):
    print(f"[PIPELINE CRITICAL ERROR]: Configured document source path '{user_configured_pdf_path}' cannot be verified.")
else:
    try:
        print("\n--- Starting Phase 1 Document Preprocessing Pipeline ---")

        # 1. Read
        raw_text_payload = extract_pdf_page_content(user_configured_pdf_path)
        # 2. Refactor Clean
        scrubbed_page_data = scrub_text_formatting(raw_text_payload)
        # 3. Structural Fragment Slicing
        runtime_document_pool = fragment_text_data(scrubbed_page_data, slice_size=500, overlap_size=50)
        # 4. Dense Space Transformations
        matrix_embeddings, runtime_encoder_instance = compile_vector_representations(runtime_document_pool)
        # 5. Build FAISS Array System
        runtime_faiss_matrix = establish_faiss_vector_index(matrix_embeddings)

        print("\n--- Phase 1 Operations Finalized Successfully! ---")
        print(f"Operational Index Capacity Count: {len(runtime_document_pool)} context fragments loaded.")

    except Exception as processing_pipeline_fault:
        print(f"\n[CRITICAL FAILURE ABORT]: Preprocessing block execution collapsed: {processing_pipeline_fault}")

## Phase 2: Interactive Inference Pipeline
Once Phase 1 is indexed and mapped, you can test your engine inside an automated chat room simulator loop. Type **`exit`** to leave the active chat session safely.

In [ ]:
# ============================================================
# PHASE 2 : CHAT INTERACTION DESK
# ============================================================
# Core Pre-flight Validation check routines
if 'user_configured_pdf_path' not in locals() or not os.path.exists(user_configured_pdf_path):
    print("[STARTUP REJECTED]: Please configure and run Phase 1 Document Ingestion cells first.")
elif 'gemini_client' not in globals() or gemini_client is None:
    print("[STARTUP REJECTED]: Missing GenAI core client authorizations. Re-execute Step 3.")
elif runtime_faiss_matrix is None or runtime_document_pool is None:
    print("[STARTUP REJECTED]: Unallocated vector arrays detected. Complete Phase 1 successfully.")
else:
    print("="*80)
    print(f"MODULAR INFRASTRUCTURE CHAT DESK ENGAGED FOR FILE: '{user_configured_pdf_path}'")
    print("Grounding Filters: Operational. Hallucination Guardrails: Active.")
    print("Submit command entry 'exit' to cleanly close communication pipelines.")
    print("="*80)

    while True:
        try:
            live_user_query = input("\nEnter your document question: ")

            if live_user_query.strip().lower() == 'exit':
                print("Gracefully disconnecting cloud operational communication loops. Goodbye!")
                break

            if not live_user_query.strip():
                print("Empty input token sequence captured. Provide a descriptive query.")
                continue

            # Operational Flow Routine execution
            # A. Match Context Nodes
            matched_evidence = query_context_store(
                search_string=live_user_query,
                database=runtime_faiss_matrix,
                resource_pool=runtime_document_pool,
                embedding_engine=runtime_encoder_instance,
                search_limit=3
            )

            # B. Compile Shielded Prompt Template
            grounded_instruction_stream = synthesis_grounded_instruction_prompt(live_user_query, matched_evidence)

            # C. Cloud Model Synthesis Pass
            grounded_llm_response = dispatch_grounded_payload(gemini_client, grounded_instruction_stream)

            # D. Render Standardized Terminal Dashboard
            print("\n" + "="*80)
            print("                         EXTRACTED EVIDENCE LOG")
            print("="*80)
            for structural_rank, node_item in enumerate(matched_evidence):
                print(f"\n[Evidence Source #{structural_rank + 1}] (Location: Page {node_item['page']} | Vector Similarity Score: {node_item['correlation']:.4f})")
                print(f"   \"{node_item['text'].strip()}\"")
                print("-" * 60)

            print("\n" + "="*80)
            print("                         GROUNDED RESPONSE")
            print("="*80)
            print(grounded_llm_response.strip())
            print("="*80 + "\n")

        except KeyboardInterrupt:
            print("\nManual interrupt signal intercepted. Terminating user application interface lines.")
            break
        except Exception as application_runtime_fault:
            print(f"\n[RUNTIME RUNTIME NOTICE]: Operational trace pass blocked: {application_runtime_fault}\n")


Manual interrupt signal intercepted. Terminating user application interface lines.


## ⚙️ Setup Instructions & Execution Guide

Follow these sequential steps to deploy and execute this advanced RAG pipeline within your environment:

### 1. Environment & Dependency Provisioning
Ensure all underlying hardware-accelerated processing dependencies are installed. Run the provisioning cell block at the top of the notebook to pull down:
* `pypdf` for parsing binary document layouts.
* `sentence-transformers` for compute-isolated local semantic vector generation.
* `faiss-cpu` for running zero-latency vectorized similarity lookups in memory.
* `google-genai` to connect seamlessly with the modern Google API system.
* `langchain-text-splitters` for structural textual parsing boundary splits.

### 2. Google Gemini API Token Configuration
* Obtain an official API Key from the Google AI Studio Dashboard.
* The system checks your background virtual machine for a preset `GEMINI_API_KEY` token block automatically.
* If no key is found, an obscured interactive passfield form will prompt you live. Paste your token safely. Output logs are wiped automatically to prevent security leaks.

### 3. Local Document Alignment
* Upload your target documentation (e.g., `Data Science.pdf` or `Week7_Project.pdf`) into your active directory workspace.
* Input the verbatim path identifier string when Phase 1 prompts you for execution context initialization.

### 4. End-to-End Pipeline Execution
* **Execute Phase 1 Exactly Once**: This compiles your text array segments, triggers mathematical vector transformations, and populates your local FAISS database map.
* **Launch Phase 2 for Chat Interactions**: Type your question strings naturally into the dynamic loop harness. The local database instantly tracks source facts, injects them into system instructions, and returns full factual sentences from Gemini.
* **Exit Protocol**: Submit the exact string value **`exit`** inside your chat terminal window to shut down open API loops cleanly.

## 🏁 Project Summary

### Project Retrospective & Core Milestones
This project successfully demonstrates the construction of a modular, fully grounded **Retrieval-Augmented Generation (RAG)** pipeline using a decoupled data processing framework. By intentionally building this pipeline step-by-step without high-level wrapper abstractions, we have explicitly mapped out the internal mechanics of state-of-the-art AI systems:

1. **Deterministic Text Extraction**: Safely parsed unstructured multi-page data blocks using `pypdf`, normalizing text fields via precise regular expression boundaries to strip formatting noise.
2. **Metadata Preservation**: Divided text frames across moving `RecursiveCharacter` windows while preserving a permanent digital trail back to original 1-indexed document coordinates.
3. **Optimized Vector Retrieval**: Populated a local FAISS index structure with dense $L_2$-normalized vector coordinates, allowing the engine to calculate exact semantic match scores via rapid inner-product computations.
4. **Context-Grounded Inference**: Engineered a strict instructional template shell to funnel verified document context records straight into **Gemini 2.5 Flash**, returning highly focused, accurate, and completely grounded responses.



## Key Learnings & Takeaways
* **The Importance of Context Cleansing**: Raw text parsing from complex PDF structures often introduces hidden spacing artifacts. Dedicated sanitization steps are essential to protect the integrity of underlying sentence embeddings.
* **Decoupling Data Processing from Inference**: Separating data preprocessing (Phase 1) from real-time question answering (Phase 2) mimics standard production software architectures. This optimization dramatically reduces user query latency, as document data is pre-indexed long before runtime questions are asked.
* **The Control Value of Strict Prompt Guardrails**: Large Language Models are naturally prone to creative fabrications when data is missing. Using clear instructions alongside hardcoded zero-temperature settings completely overrides these creative tendencies, turning the model into a reliable, literal factual extraction tool.

This modular codebase provides a clean, highly professional baseline to build secure domain assistants for notes, research documents, and enterprise files.